# 1-Implementar los endpoints

Haz una función en Python para cada uno de los endpoints del API REST:

- `GET /user/{email}`
- `GET /room/{room_id}`
- `GET /dungeon/{dungeon_id}`
- `POST /comment/`
- `DELETE /monster/{monster_id}`

Las funciones deben conectarse a la base de datos **MongoDB** y realizar las consultas pertinentes.

Se deben realizar todas las operaciones posibles directamente en la base de datos. No ejecutes cálculos en Python si no es necesario. 

In [1]:
from pymongo import MongoClient
from datetime import datetime

client = MongoClient("mongodb://localhost:27017/")
db = client["jotuns_lair"]

rooms    = db["rooms"]
users    = db["users"]
loot     = db["loot"]
monsters = db["monsters"]

## Endpoints a implementar

#### **Información de un usuario** 
````GET /user/{email} ````

Este endpoint recibe el **email** de un usuario y devuelve toda la información disponible del mismo.

Además, incluye los **20 últimos comentarios** que ha realizado ese usuario.  
De cada comentario se debe mostrar:

- **Texto**
- **Fecha de creación**
- **Categoría**
- **Id de la habitación** (`Room.IdR`)
- **Nombre de la habitación** (`Room.name`)
- **Id de la mazmorra** (`Dungeon.IdD`)
- **Nombre de la mazmorra** (`Dungeon.name`)

> Los comentarios deben ordenarse por fecha de creación, mostrando primero los más recientes.

Como en este caso tenemos que ordenar un array de comentarios, podemos hacerlo de forma nativa en MongoDB usando sortArray. Te permite ordenar un array (en este caso, el array de comentarios) por un campo específico (en este caso, la fecha de creación). 

Para más información: https://www.mongodb.com/es/docs/manual/reference/operator/aggregation/sortArray/?msockid=0d7f1cc6f6d16b8426410bc2f76a6ae1

Nota: dentro del enunciado de la práctica, dentro de hints en users, la información de la habitación se llama references_room. Sin embargo, en los JSON de Moodle, se llama referemces_rooms. Mantenemos el nombre de los JSON de Moodle, es decir, references_rooms. room_Name en realidad se escribe como room_name. 

In [2]:
def get_user(email: str):
    """
    GET /user/{email}
    Devuelve todos los campos del usuario más los 20 últimos comentarios
    con texto, fecha, categoría, id/nombre de room y id/nombre de dungeon.
    """
    pipeline = [
        # Filtrar por email
        {"$match": {"email": email}},

        # Ordenar los hints por fecha desc y quedarse con los 20 últimos 
        # TODO: preguntar si esto se aplica con lo que tenemos o con después de patrones de diseño
        {"$addFields": {
            "hints": {
                "$slice": [
                    {"$sortArray": {
                        "input": "$hints",
                        "sortBy": {"creation_date": -1}
                    }},
                    20
                ]
            }
        }},

        # Proyectar solo los campos necesarios
        {"$project": {
            "_id": 0,
            "email": 1,
            "user_name": 1,
            "creation_date": 1,
            "country": 1,
            "hints": {
                "text": 1,
                "creation_date": 1,
                "category": 1,
                "referemces_room.room_id": 1,
                "referemces_room.room_name": 1,
                "referemces_room.dungeon_id": 1,
                "referemces_room.dungeon_name": 1
            }
        }}
    ]

    result = list(users.aggregate(pipeline))
    return result[0] if result else None

In [3]:
get_user("abbottanne@example.com")

{'email': 'abbottanne@example.com',
 'hints': [{'text': 'Identify than professor statement support campaign computer.\\nEvent part half use plan. Already development front. Need today today set cold rock husband go.\\nWant safe PM front. Although born speech other project decision.\\nQuickly account staff imagine unit interest pick. Modern cultural someone appear rich quickly science. Director three red nice. True manager TV somebody school practice he.\\nRate size air off. Certain difficult drop walk cold share.\\nDraw standard prevent whole appear seat stand.',
   'category': 'lore',
   'creation_date': '2017-12-01 12:52:27.000000',
   'referemces_room': {'room_id': 667,
    'room_name': 'sticky sanctum of werewolves',
    'dungeon_id': 17,
    'dungeon_name': 'Marshgreat, Catacombs of the Wandering Degenerates'}},
  {'text': 'President result month range set specific magazine natural. Much agency bed himself production east interview. Involve yard line hear. Eat bed behind put proce

#### **Información de habitación en particular** 
````GET /room/{room_id} ````

Este endpoint recibe el **id de una habitación** y devuelve la siguiente información:

- **`idR`**
- **`name`**
- **`inWP`**
- **`outWP`**

Además, incluye:

- El **número de monstruos de cada tipo** presentes en la habitación.
- El **total de oro** que valen los tesoros de la sala.
- Los **últimos 20 comentarios** realizados sobre esa habitación.

Cada comentario debe incluir:

- **`userName`**
- **`country`**
- **`creationDate`** del usuario que lo realizó
- **Texto**
- **Fecha de publicación**
- **Categoría** del comentario

In [28]:
def get_room(room_id: int):
    """
    GET /room/{room_id}
    Devuelve todos los campos de la habitación más los 20 últimos comentarios
    con texto, fecha, categoría, id/nombre de usuario y id/nombre de dungeon.
    """
    pipeline = [
        # Filtrar por room_id
        {"$match": {"room_id": room_id}},

        {
            "$addFields": {
                "treasure_tot_value": {
                    "$sum": "$loot.gold"
                },
                "hint": {
                    "$slice": [
                        {"$sortArray": 
                         {
                            "input": "$hints",
                            "sortBy": {"creation_date": -1}
                        }
                        },
                        20
                    ]
            }
        }
        },

        {
            "$lookup": {
                "from": "rooms",
                "let": {"id_actual": "$_id"},
                "pipeline": [
                    {"$match": {"room_id": room_id}},
                    {"$unwind": "$monsters"},
                    {"$group": {
                        "_id": "$monsters.name",
                        "total": {"$sum": 1}
                        }
                    },
                    {"$project": {
                        "_id": 0,
                        "monster_type": "$_id",
                        "total": 1
                    }
                    }
                ],

                "as": "monsters_info"

            }
        },

        {"$project": {
            "_id": 0,
            "room_id": 1,
            "room_name": 1,
            "in_waypoint": 1,
            "out_waypoint": 1,
            "monsters_info": 1,
            "treasure_tot_value": 1,
            "hints": {
                "publish_by.user_name": 1,
                "publish_by.country": 1,
                "publish_by.creation_date": 1,
   
            },
            "text": 1,
            "creation_date": 1,
            "category": 17
        }
        }
    ]

    result = list(rooms.aggregate(pipeline))
    return result[0] if result else None

In [31]:
get_room(115)

{'hints': [{'publish_by': {'country': 'es_ES',
    'user_name': 'jose-ignacio63',
    'creation_date': '2022-12-16'}},
  {'publish_by': {'country': 'en_US',
    'user_name': 'joseph54',
    'creation_date': '2021-08-22'}},
  {'publish_by': {'country': 'it_IT',
    'user_name': 'marissa59',
    'creation_date': '2021-02-13'}},
  {'publish_by': {'country': 'zh_TW',
    'user_name': 'ulong',
    'creation_date': '2011-05-19'}},
  {'publish_by': {'country': 'en_US',
    'user_name': 'jonesjason',
    'creation_date': '2021-04-19'}},
  {'publish_by': {'country': 'pt_BR',
    'user_name': 'bsouza',
    'creation_date': '2011-05-29'}},
  {'publish_by': {'country': 'zh_TW',
    'user_name': 'qxue',
    'creation_date': '2016-04-30'}},
  {'publish_by': {'country': 'en_US',
    'user_name': 'cobbjoseph',
    'creation_date': '2017-11-26'}},
  {'publish_by': {'country': 'ko_KR',
    'user_name': 'hyejini',
    'creation_date': '2022-02-10'}},
  {'publish_by': {'country': 'es_ES',
    'user_name':

#### **Información de una mazmorra** 
````GET /dungeon/{dungeon_id} ````

NOTA: dentro de la base de datos, no tenemos un campo llamado lore. Por tanto, no se puede mostrar esa información.

NOTA: Interpretación del enunciado: como no se sabe si quiere las conexiones en un campo distinto o simplemente que lo tenga dentro de la habitación, entonces: 

{"$set": 
         {
             "rooms_conected": 
             {"$sortArray": 
              {
                  "input": {"$setUnion": 
                  [
                        ["$room_id"],
                        "$map": {
                            "input": "$rooms_connected",
                            "as": "room",
                            "in": "$room.room_id"
                        }
                  ]
                  },
                  "sortBy": 1
              }
             }
         }}


In [59]:
def get_dungeon(dungeon_id: int):
    """
    GET /dungeon/{dungeon_id}
    Devuelve todos los campos de la mazmorra más información del grafo interactivo.
    """

    pipeline = [
        {"$match": {"_id": dungeon_id}},
        
        {"$lookup": {
            "from": "rooms",
            "let": {"id_actual": "$_id"},
            "pipeline": [
                {"$match": {"_id": dungeon_id}},

                {
                    "$lookup": {
                        "from": "rooms",
                        "let": {"id_actual": "$_id"},
                        "pipeline": [
                            {"$match": {"_id": dungeon_id}},
                            {"$unwind": "$hints"},
                            {"$group": {
                                "_id": "$hints.category",
                                "total": {"$sum": 1}
                            }},
                            {"$project": {
                                "_id": 0,
                                "hint_category": "$_id",
                                "total": 1
                            }}
                        ],
                        "as": "hints_info"
                    }
                },

                {"$project": {
                "_id": 0,
                "room_id": 1,
                "room_name": 1,
                "rooms_connected": {
                    "$map": {
                        "input": "$rooms_connected",
                        "as": "r",
                        "in": {"room_id": "$$r.room_id"}
                    }
                },
                "monsters": {
                    "$map": {
                        "input": "$monsters",
                        "as": "m",
                        "in": {"id": "$$m.id", "name": "$$m.name"}
                    }
                },
                "loot": {
                    "$map": {
                        "input": {"$setUnion": "$loot"},
                        "as": "l",
                        "in": {"id": "$$l.id", "name": "$$l.name"}
                    }
                },
                "hints_info": 1
            }}
            ], 
            "as": "rooms_info"
        }},

        {"$project": {
            "_id": 0,
            "dungeon_id": 1,
            "dungeon_name": 1,
            "rooms_info": 1
        }}
        
    ]

    result = list(db["rooms"].aggregate(pipeline))
    return result[0] if result else None

In [60]:
from bson.objectid import ObjectId

get_dungeon(ObjectId('69df421577db15125cd105bd'))

{'dungeon_id': 0,
 'dungeon_name': 'Burghap, Prison of the Jealous Hippies',
 'rooms_info': [{'room_id': 1,
   'room_name': 'pantry of otaku',
   'hints_info': [{'total': 4, 'hint_category': 'bug'},
    {'total': 10, 'hint_category': 'lore'},
    {'total': 1, 'hint_category': 'hint'}],
   'rooms_connected': [{'room_id': 18}],
   'monsters': [{'id': 25, 'name': 'ghost'},
    {'id': 28, 'name': 'gibbering mouther'},
    {'id': 48, 'name': 'mimic'}],
   'loot': [{'id': 29, 'name': 'Birdpipes'},
    {'id': 91, 'name': "Cobbler's Tools"},
    {'id': 97, 'name': 'Crossbow Bolts'},
    {'id': 101, 'name': 'Crowbar'},
    {'id': 165, 'name': 'Herbalism Kit'},
    {'id': 199, 'name': 'Longhorn'},
    {'id': 225, 'name': 'Net'},
    {'id': 233, 'name': 'Padded'},
    {'id': 342, 'name': 'Thelarr'},
    {'id': 374, 'name': "Woodcarver's Tools"}]}]}

### Información de una mazmorra  
`GET /dungeon/{dungeon_id}`

Este endpoint recibe el identificador de una mazmorra y devuelve información clave de una mazmorra del juego.

Debe incluir

- **Datos principales de la mazmorra**:
    - `idM`
    - `name`
    - `lore`

- **Datos para el grafo interactivo**:
    1. **Habitaciones de la mazmorra**: `id` y `name` de cada habitación.
    2. **Conexiones entre habitaciones** dentro de la mazmorra.
    3. **Monstruos por habitación**: `id` y `name` de cada monstruo.
    4. **Tesoros por habitación**: `id` y `name` de cada tesoro.
    5. **Comentarios por categoría en cada habitación**: número total por categoría.

#### **Publicar comentario** 
````POST /comment/```` 

Este endpoint añade un nuevo comentario. Recibe como parámetros: user_email (str), room_id (int), 
text (str), category (str).

In [66]:
def post_comment(user_email: str, room_id: int, text: str, category: str) -> dict:
    """POST /comment/: añade un comentario en el array hints de una habitación."""
    # 1) Validar usuario
    user = users.find_one({"email": user_email})
    if not user:
        return {"ok": False, "error": "User not found"}

    # 2) Construir comentario
    comment = {
        "text": text,
        "creation_date": datetime.utcnow(),
        "category": category,
        "publish_by": {
            "user_id": user["_id"],
            "user_name": user.get("user_name"),
            "country": user.get("country"),
            "creation_date": user.get("creation_date")
        }
    }

    # 3) Insertar en la habitación (push al array hints)
    result_room = rooms.update_one(
        {"room_id": room_id},
        {"$push": {"hints": comment}}
    )


    if result_room.matched_count == 0:
        return {"ok": False, "error": "Room not found"}

    if result_room.modified_count == 0:
        return {"ok": False, "error": "Comment was not inserted"}


    result_user = users.update_one(
        {"email": user_email},
        {"$push": {"hints": {
            "text": text,
            "creation_date": datetime.utcnow(),
            "category": category,
            "referemces_room": {
                "room_id": room_id
            }
        }}})

    if result_user.matched_count == 0:
        return {"ok": False, "error": "User not found"}
    
    if result_user.modified_count == 0:
        return {"ok": False, "error": "Comment was not inserted in user"}


    return {
        "ok": True,
        "message": "Comment posted successfully",
        "room_id": room_id,
        "user_email": user_email
    }

In [67]:
# Prueba del endpoint POST /comment/
post_comment("abbottanne@example.com", 115, "prueba", "positive")

{'ok': True,
 'message': 'Comment posted successfully',
 'room_id': 115,
 'user_email': 'abbottanne@example.com'}

#### **Borrar monstruo** 
````DELETE /monsters/{monster_id}```` 

Este endpoint recibe el id de un monstruo y lo elimina de la base de datos.

In [ ]:
def delete_monster(monster_id: int) -> dict:
    """DELETE /monster/: elimina un monstruo de todas las habitaciones."""
    result_rooms = rooms.update_many(
        {"monsters.id": monster_id},
        {"$pull": {"monsters": {"id": monster_id}}}
    )

    if result_rooms.matched_count == 0:
        return {"ok": False, "error": "Monster not found in any room"}

    result_monster = monsters.delete_one({"id": monster_id})
    if result_monster.deleted_count == 0:
        return {"ok": False, "error": "Monster not found in monsters collection"}

    return {
        "ok": True,
        "message": f"Monster with id {monster_id} deleted from {result_rooms.modified_count} rooms"
    }